In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nolanbconaway/pitchfork-data")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'pitchfork-data' dataset.
Path to dataset files: /kaggle/input/pitchfork-data


In [18]:
import sqlite3
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime

np.random.seed(42)
random.seed(42)

# Connect to SOURCE database
src_conn = sqlite3.connect("/root/.cache/kagglehub/datasets/nolanbconaway/pitchfork-data/versions/1/database.sqlite")

# Pull core data from source
reviews_src = pd.read_sql_query("SELECT * FROM reviews",  src_conn)
genres_src  = pd.read_sql_query("SELECT * FROM genres",   src_conn)
labels_src  = pd.read_sql_query("SELECT * FROM labels",   src_conn)
artists_src = pd.read_sql_query("SELECT * FROM artists",  src_conn)
years_src   = pd.read_sql_query("SELECT * FROM years",    src_conn)
content_src = pd.read_sql_query("SELECT * FROM content",  src_conn)
src_conn.close()

print(f"Source reviews : {len(reviews_src):,} rows")
print(f"Source genres  : {len(genres_src):,} rows")
print(f"Source labels  : {len(labels_src):,} rows")
print(f"Source artists : {len(artists_src):,} rows")
print(reviews_src.head(3))

Source reviews : 18,393 rows
Source genres  : 22,680 rows
Source labels  : 20,190 rows
Source artists : 18,831 rows
   reviewid                 title          artist  \
0     22703             mezzanine  massive attack   
1     22721          prelapsarian        krallice   
2     22659  all of them naturals    uranium club   

                                                 url  score  best_new_music  \
0  http://pitchfork.com/reviews/albums/22703-mezz...    9.3               0   
1  http://pitchfork.com/reviews/albums/22721-prel...    7.9               0   
2  http://pitchfork.com/reviews/albums/22659-all-...    7.3               0   

           author  author_type    pub_date  pub_weekday  pub_day  pub_month  \
0     nate patrin  contributor  2017-01-08            6        8          1   
1        zoe camp  contributor  2017-01-07            5        7          1   
2  david glickman  contributor  2017-01-07            5        7          1   

   pub_year  
0      2017  
1      20

In [19]:
# Create / recreate the analytics database
DB_PATH = "pitchfork_analytics.sqlite"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

ana_conn = sqlite3.connect(DB_PATH)
cur = ana_conn.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

# 1. dim_genre
cur.execute("""
CREATE TABLE dim_genre (
    genre_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    genre_name        TEXT    NOT NULL UNIQUE,          -- NOMINAL   (category)
    genre_category    TEXT    NOT NULL,                 -- NOMINAL   (broad group)
    popularity_tier   TEXT    NOT NULL                  -- ORDINAL   (ordered quality)
        CHECK(popularity_tier IN ('Niche','Underground','Emerging','Mainstream','Dominant')),
    genre_avg_score   REAL    NOT NULL                  -- RATIO     (0-10, meaningful zero)
        CHECK(genre_avg_score >= 0 AND genre_avg_score <= 10)
);
""")

# 2. dim_label
cur.execute("""
CREATE TABLE dim_label (
    label_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    label_name        TEXT    NOT NULL UNIQUE,          -- NOMINAL
    label_type        TEXT    NOT NULL                  -- ORDINAL (size/prestige tier)
        CHECK(label_type IN ('Micro-indie','Indie','Mid-size','Major')),
    country           TEXT    NOT NULL,                 -- NOMINAL
    founded_year      INTEGER NOT NULL,                 -- INTERVAL (year; no meaningful zero)
    total_releases    INTEGER NOT NULL DEFAULT 0        -- RATIO    (count ≥ 0)
        CHECK(total_releases >= 0)
);
""")

# 3. dim_artist
cur.execute("""
CREATE TABLE dim_artist (
    artist_id             INTEGER PRIMARY KEY AUTOINCREMENT,
    artist_name           TEXT    NOT NULL UNIQUE,       -- NOMINAL
    country               TEXT    NOT NULL,              -- NOMINAL
    career_start_year     INTEGER NOT NULL,              -- INTERVAL (year scale)
    total_albums_reviewed INTEGER NOT NULL DEFAULT 0     -- RATIO    (count ≥ 0)
        CHECK(total_albums_reviewed >= 0),
    avg_score             REAL    NOT NULL               -- RATIO    (0-10)
        CHECK(avg_score >= 0 AND avg_score <= 10)
);
""")

# 4. fact_reviews (≥ 1000 rows)
cur.execute("""
CREATE TABLE fact_reviews (
    reviewid          INTEGER PRIMARY KEY,              -- from source
    title             TEXT    NOT NULL,                 -- NOMINAL
    artist_id         INTEGER NOT NULL REFERENCES dim_artist(artist_id),
    genre_id          INTEGER NOT NULL REFERENCES dim_genre(genre_id),
    label_id          INTEGER NOT NULL REFERENCES dim_label(label_id),
    score             REAL    NOT NULL                  -- RATIO (0-10, meaningful zero)
        CHECK(score >= 0 AND score <= 10),
    rating_category   TEXT    NOT NULL                  -- ORDINAL
        CHECK(rating_category IN ('Poor','Below Average','Average','Good','Great','Excellent')),
    best_new_music     INTEGER NOT NULL DEFAULT 0       -- NOMINAL flag (0/1)
        CHECK(best_new_music IN (0,1)),
    pub_year          INTEGER NOT NULL,                 -- INTERVAL (calendar year)
    pub_month         INTEGER NOT NULL                  -- INTERVAL (1-12)
        CHECK(pub_month BETWEEN 1 AND 12),
    word_count        INTEGER NOT NULL                  -- RATIO (count of review words ≥ 0)
        CHECK(word_count >= 0),
    sentiment_score   REAL    NOT NULL                  -- INTERVAL (-1 to +1, arbitrary zero)
        CHECK(sentiment_score BETWEEN -1.0 AND 1.0),
    author_experience TEXT    NOT NULL                  -- ORDINAL
        CHECK(author_experience IN ('Junior','Mid-level','Senior','Lead'))
);
""")

# 5. review_engagement  (COMPOUND PRIMARY KEY)
cur.execute("""
CREATE TABLE review_engagement (
    reviewid          INTEGER NOT NULL REFERENCES fact_reviews(reviewid),
    platform          TEXT    NOT NULL                  -- NOMINAL
        CHECK(platform IN ('Web','Mobile','App','RSS')),
    views             INTEGER NOT NULL DEFAULT 0        -- RATIO (≥ 0)
        CHECK(views >= 0),
    likes             INTEGER NOT NULL DEFAULT 0        -- RATIO
        CHECK(likes >= 0),
    shares            INTEGER NOT NULL DEFAULT 0        -- RATIO
        CHECK(shares >= 0),
    engagement_tier   TEXT    NOT NULL                  -- ORDINAL
        CHECK(engagement_tier IN ('Low','Medium','High','Viral')),
    PRIMARY KEY (reviewid, platform)                    -- COMPOUND KEY
);
""")

ana_conn.commit()
print("Schema created successfully.")
print("Tables:", [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()])

Schema created successfully.
Tables: ['dim_genre', 'sqlite_sequence', 'dim_label', 'dim_artist', 'fact_reviews', 'review_engagement']


In [20]:
# Step 3: Populate Dimension Tables
ORDINAL_TIERS  = ['Niche','Underground','Emerging','Mainstream','Dominant']
LABEL_TYPES    = ['Micro-indie','Indie','Mid-size','Major']
EXPERIENCE_LVL = ['Junior','Mid-level','Senior','Lead']
PLATFORMS      = ['Web','Mobile','App','RSS']
ENGAGEMENT_T   = ['Low','Medium','High','Viral']
COUNTRIES      = ['USA','UK','Canada','Australia','Germany','France',
                  'Japan','Sweden','Norway','Netherlands','Brazil','Ireland']

# dim_genre: derive unique genres from source + assign randomised attributes
unique_genres = genres_src['genre'].dropna().unique().tolist()
genre_categories = {
    'electronic':   'Electronic/Dance',
    'rock':         'Rock/Alternative',
    'experimental': 'Avant-garde',
    'rap':          'Hip-hop/Rap',
    'pop/r&b':      'Pop & Soul',
    'folk/country': 'Folk & Country',
    'jazz':         'Jazz & Classical',
    'metal':        'Rock/Alternative',
    'global':       'World Music',
}
genre_rows = []
for g in unique_genres:
    cat   = genre_categories.get(g, 'Other')
    tier  = random.choice(ORDINAL_TIERS)
    gscore = round(np.random.uniform(5.5, 8.5), 2)   # RATIO 0-10
    genre_rows.append((g, cat, tier, gscore))

cur.executemany(
    "INSERT INTO dim_genre(genre_name, genre_category, popularity_tier, genre_avg_score) VALUES(?,?,?,?)",
    genre_rows
)

# dim_label: derive unique labels from source
unique_labels = labels_src['label'].dropna().unique().tolist()
label_rows = []
for lbl in unique_labels[:500]:   # cap at 500 to keep it manageable
    ltype  = random.choice(LABEL_TYPES)
    cty    = random.choice(COUNTRIES)
    yr     = random.randint(1950, 2010)   # INTERVAL
    rels   = random.randint(1, 500)       # RATIO
    label_rows.append((lbl, ltype, cty, yr, rels))

cur.executemany(
    "INSERT INTO dim_label(label_name, label_type, country, founded_year, total_releases) VALUES(?,?,?,?,?)",
    label_rows
)

# dim_artist: derive unique artists from source
unique_artists = artists_src['artist'].dropna().unique().tolist()
artist_rows = []
for art in unique_artists[:3000]:
    cty      = random.choice(COUNTRIES)
    start_yr = random.randint(1960, 2015)   # INTERVAL
    n_albums = random.randint(1, 20)        # RATIO
    a_score  = round(np.random.uniform(4.0, 9.5), 2)  # RATIO
    artist_rows.append((art, cty, start_yr, n_albums, a_score))

cur.executemany(
    "INSERT INTO dim_artist(artist_name, country, career_start_year, total_albums_reviewed, avg_score) VALUES(?,?,?,?,?)",
    artist_rows
)

ana_conn.commit()

# Fetch lookup maps for FK assignment
genre_map  = {r[0]: r[1] for r in cur.execute("SELECT genre_name, genre_id FROM dim_genre").fetchall()}
label_map  = {r[0]: r[1] for r in cur.execute("SELECT label_name, label_id FROM dim_label").fetchall()}
artist_map = {r[0]: r[1] for r in cur.execute("SELECT artist_name, artist_id FROM dim_artist").fetchall()}

genre_ids  = list(genre_map.values())
label_ids  = list(label_map.values())
artist_ids = list(artist_map.values())

print(f"dim_genre  : {cur.execute('SELECT COUNT(*) FROM dim_genre').fetchone()[0]} rows")
print(f"dim_label  : {cur.execute('SELECT COUNT(*) FROM dim_label').fetchone()[0]} rows")
print(f"dim_artist : {cur.execute('SELECT COUNT(*) FROM dim_artist').fetchone()[0]} rows")

dim_genre  : 9 rows
dim_label  : 500 rows
dim_artist : 3000 rows


In [21]:
# Step 4: Populate fact_reviews (using real source data, ≥ 1000 rows)

def score_to_category(s):
    """Map numeric score → ordinal rating category."""
    if   s < 3.0: return 'Poor'
    elif s < 5.0: return 'Below Average'
    elif s < 6.5: return 'Average'
    elif s < 7.5: return 'Good'
    elif s < 8.5: return 'Great'
    else:          return 'Excellent'

# Merge source tables to build a rich fact frame
merged = (
    reviews_src
    .merge(genres_src.drop_duplicates('reviewid'), on='reviewid', how='left')
    .merge(labels_src.drop_duplicates('reviewid'), on='reviewid', how='left')
    .merge(years_src .drop_duplicates('reviewid'), on='reviewid', how='left')
    .merge(content_src,                             on='reviewid', how='left')
)

# Keep only rows where genre/label/artist are in our dimension tables
merged = merged[
    merged['genre'].isin(genre_map) &
    merged['label'].isin(label_map) &
    merged['artist'].isin(artist_map)
].drop_duplicates('reviewid').reset_index(drop=True)

# Randomly augment extra columns (reproducible via seed already set)
merged['word_count']      = [len(str(c).split()) if pd.notna(c) else random.randint(100, 900)
                              for c in merged['content']]              # RATIO
merged['sentiment_score'] = np.round(
    np.random.uniform(-1.0, 1.0, len(merged)), 4)                     # INTERVAL
merged['author_experience'] = np.random.choice(
    EXPERIENCE_LVL, len(merged), p=[0.15, 0.40, 0.35, 0.10])         # ORDINAL

# Build insert rows
fact_rows = []
for _, row in merged.iterrows():
    gid  = genre_map.get(row['genre'])
    lid  = label_map.get(row['label'])
    aid  = artist_map.get(row['artist'])
    sc   = float(row['score']) if pd.notna(row['score']) else round(random.uniform(3, 9), 1)
    cat  = score_to_category(sc)
    bnm  = int(row['best_new_music']) if pd.notna(row['best_new_music']) else 0
    yr   = int(row['pub_year'])  if pd.notna(row['pub_year'])  else random.randint(1999, 2017)
    mo   = int(row['pub_month']) if pd.notna(row['pub_month']) else random.randint(1, 12)
    wc   = int(row['word_count'])
    sent = float(row['sentiment_score'])
    exp  = row['author_experience']
    fact_rows.append((row['reviewid'], row['title'], aid, gid, lid,
                      sc, cat, bnm, yr, mo, wc, sent, exp))

cur.executemany("""
    INSERT INTO fact_reviews
    (reviewid, title, artist_id, genre_id, label_id, score, rating_category,
     best_new_music, pub_year, pub_month, word_count, sentiment_score, author_experience)
    VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
""", fact_rows)

ana_conn.commit()
row_count = cur.execute("SELECT COUNT(*) FROM fact_reviews").fetchone()[0]
print(f"fact_reviews : {row_count:,} rows  (requirement: ≥ 1000)" if row_count >= 1000
      else f"fact_reviews : {row_count:,} rows  ← BELOW 1000!")

# Step 5: Populate review_engagement with compound PK
review_ids = [r[0] for r in cur.execute("SELECT reviewid FROM fact_reviews").fetchall()]

engagement_rows = []
for rid in review_ids:
    platforms_assigned = random.sample(PLATFORMS, k=random.randint(1, 3))
    for plat in platforms_assigned:
        views  = int(np.random.lognormal(7, 1.5))       # RATIO – right-skewed traffic
        likes  = int(views * np.random.uniform(0.01, 0.20))
        shares = int(likes  * np.random.uniform(0.0,  0.30))
        if   views > 50_000: tier = 'Viral'
        elif views > 10_000: tier = 'High'
        elif views >  2_000: tier = 'Medium'
        else:                 tier = 'Low'
        engagement_rows.append((rid, plat, views, likes, shares, tier))

cur.executemany("""
    INSERT INTO review_engagement(reviewid, platform, views, likes, shares, engagement_tier)
    VALUES (?,?,?,?,?,?)
""", engagement_rows)

ana_conn.commit()
eng_count = cur.execute("SELECT COUNT(*) FROM review_engagement").fetchone()[0]
print(f"review_engagement : {eng_count:,} rows  (compound PK on reviewid+platform)")

fact_reviews : 4,866 rows  (requirement: ≥ 1000)
review_engagement : 9,627 rows  (compound PK on reviewid+platform)


In [22]:
# Verify table structure and retrieve data
print("Database knowledge: schema inspection")

schema_q = """
    SELECT m.name AS table_name,
           COUNT(p.name) AS column_count
    FROM   sqlite_master m
    JOIN   pragma_table_info(m.name) p ON 1=1
    WHERE  m.type = 'table'
    GROUP  BY m.name
"""
df_schema = pd.read_sql_query(schema_q, ana_conn)
print(df_schema.to_string(index=False))

print("Data retrieval: top-10 highest-scored reviews")
top10 = pd.read_sql_query("""
    SELECT fr.reviewid,
           fr.title,
           da.artist_name,
           dg.genre_name,
           dl.label_name,
           fr.score,                     -- RATIO column
           fr.rating_category,           -- ORDINAL column
           fr.pub_year,                  -- INTERVAL column
           fr.word_count,                -- RATIO column
           fr.sentiment_score            -- INTERVAL column
    FROM   fact_reviews   fr
    JOIN   dim_artist da ON da.artist_id = fr.artist_id
    JOIN   dim_genre  dg ON dg.genre_id  = fr.genre_id
    JOIN   dim_label  dl ON dl.label_id  = fr.label_id
    ORDER  BY fr.score DESC
    LIMIT  10
""", ana_conn)
display(top10)

print("Genre distribution (nominal column)")
genre_dist = pd.read_sql_query("""
    SELECT dg.genre_name,
           dg.popularity_tier,           -- ORDINAL
           COUNT(*)          AS reviews,
           ROUND(AVG(fr.score), 3) AS avg_score,
           ROUND(MIN(fr.score), 1) AS min_score,
           ROUND(MAX(fr.score), 1) AS max_score
    FROM   fact_reviews  fr
    JOIN   dim_genre     dg ON dg.genre_id = fr.genre_id
    GROUP  BY dg.genre_name
    ORDER  BY reviews DESC
""", ana_conn)
display(genre_dist)

Database knowledge: schema inspection
       table_name  column_count
       dim_artist             6
        dim_genre             5
        dim_label             6
     fact_reviews            13
review_engagement             6
  sqlite_sequence             2
Data retrieval: top-10 highest-scored reviews


,reviewid,title,artist_name,genre_name,label_name,score,rating_category,pub_year,word_count,sentiment_score
0,178,source tags and codes,...and you will know us by the trail of dead,rock,interscope,10.0,Excellent,2002,1085,-0.8552
1,838,music has the right to children,boards of canada,electronic,warp,10.0,Excellent,2004,844,-0.3461
2,6200,"crooked rain, crooked rain: la's desert origins",pavement,rock,matador,10.0,Excellent,2004,1394,-0.4643
3,6307,animals,pink floyd,rock,columbia,10.0,Excellent,2000,384,-0.2912
4,6656,kid a,radiohead,rock,capitol,10.0,Excellent,2000,1238,-0.4731
5,7728,born to run: 30th anniversary edition,bruce springsteen,rock,columbia,10.0,Excellent,2005,1048,0.6849
6,8676,yankee hotel foxtrot,wilco,rock,nonesuch,10.0,Excellent,2002,1186,-0.9017
7,11376,otis blue: otis redding sings soul [collector'...,otis redding,pop/r&b,rhino,10.0,Excellent,2008,1359,-0.3460
8,11866,pink flag,wire,rock,harvest,10.0,Excellent,2006,844,0.7186
9,13130,reckoning [deluxe edition],r.e.m.,rock,universal,10.0,Excellent,2009,708,-0.8292


Genre distribution (nominal column)


,genre_name,popularity_tier,reviews,avg_score,min_score,max_score
0,rock,Emerging,2506,7.210,0.4,10.0
1,electronic,Niche,728,7.140,2.0,10.0
2,rap,Underground,562,6.998,1.6,10.0
3,pop/r&b,Underground,383,7.059,2.5,10.0
4,experimental,Underground,236,7.590,3.5,10.0
5,folk/country,Niche,180,7.562,3.0,9.7
6,metal,Niche,164,7.419,3.9,9.0
7,jazz,Dominant,61,7.813,3.5,10.0
8,global,Niche,46,7.593,4.4,9.4


In [23]:
# Advanced extraction – subquery, window function, compound key join
print("Advanced extraction: subquery & window functions")
above_avg = pd.read_sql_query("""
    SELECT fr.title,
           da.artist_name,
           dg.genre_name,
           fr.score,
           fr.rating_category,
           fr.pub_year,
           fr.author_experience,
           -- window: rank within genre by score
           RANK() OVER (PARTITION BY fr.genre_id ORDER BY fr.score DESC) AS genre_rank
    FROM   fact_reviews fr
    JOIN   dim_artist da ON da.artist_id = fr.artist_id
    JOIN   dim_genre  dg ON dg.genre_id  = fr.genre_id
    WHERE  fr.score > (SELECT AVG(score) FROM fact_reviews)
    ORDER  BY fr.score DESC
    LIMIT  20
""", ana_conn)
display(above_avg)

print("Compound key table: engagement by platform")
# Join via compound PK (reviewid, platform)
engagement_summary = pd.read_sql_query("""
    SELECT re.platform,                          -- NOMINAL
           re.engagement_tier,                   -- ORDINAL
           COUNT(*)                AS records,
           SUM(re.views)           AS total_views,   -- RATIO
           ROUND(AVG(re.views),0)  AS avg_views,     -- RATIO
           SUM(re.likes)           AS total_likes,
           SUM(re.shares)          AS total_shares
    FROM   review_engagement re
    GROUP  BY re.platform, re.engagement_tier
    ORDER  BY re.platform, re.engagement_tier
""", ana_conn)
display(engagement_summary)

print("Label-type performance (ordinal label_type)")
label_perf = pd.read_sql_query("""
    SELECT dl.label_type,                        -- ORDINAL
           dl.country,                           -- NOMINAL
           COUNT(fr.reviewid) AS total_reviews,
           ROUND(AVG(fr.score), 2) AS avg_score, -- RATIO
           SUM(fr.best_new_music)  AS bnm_count  -- RATIO
    FROM   fact_reviews fr
    JOIN   dim_label dl ON dl.label_id = fr.label_id
    GROUP  BY dl.label_type, dl.country
    ORDER  BY dl.label_type, avg_score DESC
    LIMIT  20
""", ana_conn)
display(label_perf)

Advanced extraction: subquery & window functions


,title,artist_name,genre_name,score,rating_category,pub_year,author_experience,genre_rank
0,music has the right to children,boards of canada,electronic,10.0,Excellent,2004,Mid-level,1
1,source tags and codes,...and you will know us by the trail of dead,rock,10.0,Excellent,2002,Senior,1
2,"crooked rain, crooked rain: la's desert origins",pavement,rock,10.0,Excellent,2004,Junior,1
3,animals,pink floyd,rock,10.0,Excellent,2000,Senior,1
4,kid a,radiohead,rock,10.0,Excellent,2000,Mid-level,1
5,born to run: 30th anniversary edition,bruce springsteen,rock,10.0,Excellent,2005,Mid-level,1
6,yankee hotel foxtrot,wilco,rock,10.0,Excellent,2002,Lead,1
7,pink flag,wire,rock,10.0,Excellent,2006,Junior,1
8,reckoning [deluxe edition],r.e.m.,rock,10.0,Excellent,2009,Mid-level,1
9,kid a: special collectors edition,radiohead,rock,10.0,Excellent,2009,Mid-level,1


Compound key table: engagement by platform


,platform,engagement_tier,records,total_views,avg_views,total_likes,total_shares
0,App,High,181,3465134,19144.0,349164,54698
1,App,Low,1548,1034954,669.0,108491,15372
2,App,Medium,630,2771648,4399.0,286268,40350
3,App,Viral,14,1913757,136697.0,183503,29868
4,Mobile,High,167,2947806,17652.0,298583,44470
5,Mobile,Low,1562,1075404,688.0,114571,16333
6,Mobile,Medium,684,2958519,4325.0,315204,48181
7,Mobile,Viral,10,898525,89853.0,122385,10822
8,RSS,High,147,2712936,18455.0,300911,46186
9,RSS,Low,1605,1113141,694.0,111042,16712


Label-type performance (ordinal label_type)


,label_type,country,total_reviews,avg_score,bnm_count
0,Indie,Canada,140,7.48,13
1,Indie,USA,56,7.39,5
2,Indie,France,43,7.39,3
3,Indie,Sweden,31,7.35,0
4,Indie,Brazil,248,7.33,33
5,Indie,Australia,44,7.28,7
6,Indie,Germany,46,7.21,6
7,Indie,Netherlands,133,7.19,15
8,Indie,Ireland,112,7.12,6
9,Indie,Norway,69,7.07,11


In [24]:
# Association rules – co-occurrence of genre + rating_category
print("Association rules: genre × rating_category patterns")

# Co-occurrence counts
cooccur = pd.read_sql_query("""
    SELECT dg.genre_name         AS antecedent,
           fr.rating_category    AS consequent,
           COUNT(*)              AS support_count
    FROM   fact_reviews fr
    JOIN   dim_genre dg ON dg.genre_id = fr.genre_id
    GROUP  BY dg.genre_name, fr.rating_category
    ORDER  BY support_count DESC
""", ana_conn)

# Total per genre (denominator for confidence)
genre_totals = cooccur.groupby('antecedent')['support_count'].sum().rename('genre_total')
cooccur = cooccur.join(genre_totals, on='antecedent')

total_reviews = cooccur['support_count'].sum()
cooccur['support']    = (cooccur['support_count'] / total_reviews).round(4)
cooccur['confidence'] = (cooccur['support_count'] / cooccur['genre_total']).round(4)

# Consequent frequency (for lift)
cons_freq = cooccur.groupby('consequent')['support_count'].sum() / total_reviews
cooccur['cons_support'] = cooccur['consequent'].map(cons_freq)
cooccur['lift']         = (cooccur['confidence'] / cooccur['cons_support']).round(3)

# Show rules with lift > 1.0 (positive association)
strong_rules = cooccur[cooccur['lift'] > 1.0].sort_values('lift', ascending=False).head(20)
display(strong_rules[['antecedent','consequent','support','confidence','lift']])

Association rules: genre × rating_category patterns


,antecedent,consequent,support,confidence,lift
32,jazz,Excellent,0.0037,0.2951,2.528
36,global,Excellent,0.0021,0.2174,1.862
9,experimental,Great,0.0255,0.5254,1.502
14,folk/country,Great,0.0181,0.4889,1.398
15,metal,Great,0.0164,0.4878,1.395
28,rock,Poor,0.0045,0.0088,1.381
10,rap,Average,0.0245,0.2117,1.350
16,pop/r&b,Average,0.0164,0.2089,1.332
24,rap,Below Average,0.0058,0.0498,1.206
31,pop/r&b,Below Average,0.0039,0.0496,1.201


In [25]:
# Data preparation – ingest from DB, encode, normalise
print("Data preparation for onward processing")

# Pull the full fact table with all dimension attributes
df_full = pd.read_sql_query("""
    SELECT fr.reviewid,
           fr.title,
           da.artist_name,
           da.country             AS artist_country,    -- NOMINAL
           dg.genre_name,                               -- NOMINAL
           dg.popularity_tier,                          -- ORDINAL
           dl.label_name,
           dl.label_type,                               -- ORDINAL
           fr.score,                                    -- RATIO
           fr.rating_category,                          -- ORDINAL
           fr.best_new_music,                           -- NOMINAL flag
           fr.pub_year,                                 -- INTERVAL
           fr.pub_month,                                -- INTERVAL
           fr.word_count,                               -- RATIO
           fr.sentiment_score,                          -- INTERVAL
           fr.author_experience                         -- ORDINAL
    FROM   fact_reviews   fr
    JOIN   dim_artist da ON da.artist_id = fr.artist_id
    JOIN   dim_genre  dg ON dg.genre_id  = fr.genre_id
    JOIN   dim_label  dl ON dl.label_id  = fr.label_id
""", ana_conn)

print(f"Raw shape: {df_full.shape}")
print(f"\nNull counts:\n{df_full.isnull().sum()[df_full.isnull().sum() > 0]}")

# Ordinal encoding
ordinal_maps = {
    'popularity_tier':   {'Niche':1,'Underground':2,'Emerging':3,'Mainstream':4,'Dominant':5},
    'label_type':        {'Micro-indie':1,'Indie':2,'Mid-size':3,'Major':4},
    'rating_category':   {'Poor':1,'Below Average':2,'Average':3,'Good':4,'Great':5,'Excellent':6},
    'author_experience': {'Junior':1,'Mid-level':2,'Senior':3,'Lead':4},
}
for col, mapping in ordinal_maps.items():
    df_full[col + '_enc'] = df_full[col].map(mapping)

# Min-max normalisation on ratio columns
for col in ['score', 'word_count']:
    mn, mx = df_full[col].min(), df_full[col].max()
    df_full[col + '_norm'] = ((df_full[col] - mn) / (mx - mn)).round(4)

# One-hot encode nominal: genre_name
genre_dummies = pd.get_dummies(df_full['genre_name'], prefix='genre').astype(int)
df_prepared   = pd.concat([df_full, genre_dummies], axis=1)

print(f"\nPrepared shape: {df_prepared.shape}")
print(f"\nSample prepared rows:")
display(df_prepared[[
    'title','genre_name','score','score_norm',
    'rating_category','rating_category_enc',
    'pub_year','sentiment_score','word_count_norm'
]].head(10))

# Close connection
ana_conn.close()
print(f"\nDatabase '{DB_PATH}' saved and connection closed.")
print("\nAll 5 learning outcomes demonstrated")

Data preparation for onward processing
Raw shape: (4866, 16)

Null counts:
Series([], dtype: int64)

Prepared shape: (4866, 31)

Sample prepared rows:


,title,genre_name,score,score_norm,rating_category,rating_category_enc,pub_year,sentiment_score,word_count_norm
0,heartbreaker,rock,9.0,0.8958,Excellent,6,2000,-0.7694,0.2245
1,the virgin suicides,electronic,7.2,0.7083,Good,4,2000,0.8882,0.1312
2,"10,000 hz legend",electronic,7.6,0.7500,Great,5,2001,0.3188,0.2611
3,talkie walkie,electronic,8.3,0.8229,Great,5,2004,-0.8776,0.1478
4,democrazy,pop/r&b,3.2,0.2917,Below Average,2,2004,0.1708,0.1293
5,source tags and codes,rock,10.0,1.0000,Excellent,6,2002,-0.8552,0.2942
6,the secret of elena's tomb ep,rock,6.7,0.6562,Good,4,2003,0.5013,0.2543
7,relative ways ep,rock,8.6,0.8542,Excellent,6,2001,0.2677,0.1586
8,worlds apart,rock,4.0,0.3750,Below Average,2,2005,-0.2646,0.2061
9,drukqs,electronic,5.5,0.5312,Average,3,2001,0.4719,0.2324



Database 'pitchfork_analytics.sqlite' saved and connection closed.

All 5 learning outcomes demonstrated
